In [ ]:
# NLP and data processing
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from itertools import combinations


In [ ]:
df=pd.read_csv("/content/bbc_news.csv")

In [ ]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from string import punctuation
import nltk
import string
from nltk.stem import WordNetLemmatizer # Import WordNetLemmatizer
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet') # Download the wordnet corpus
nltk.download('punkt_tab') # Download punkt_tab
lemmatizer = WordNetLemmatizer() # Initialize the lemmatizer
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    text=text.lower()
    text=text.translate(str.maketrans('', '', string.punctuation)) # Remove punctuation
    text=re.sub(r'[^a-zA-Z0-9\s]', '', text)
    tokens=word_tokenize(text)#tokenization
    stop_words=set(stopwords.words('english'))#stop words
    tokens=[word for word in tokens if word not in stop_words]
    tokens = [lemmatizer.lemmatize(word) for word in tokens]  # Lemmatization
    return " ".join(tokens)
df["cleaned_text"]=df["description"].astype(str).apply(preprocess_text)
print(df)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


                                                   title  \
0      Ukraine: Angry Zelensky vows to punish Russian...   
1      War in Ukraine: Taking cover in a town under a...   
2             Ukraine war 'catastrophic for global food'   
3      Manchester Arena bombing: Saffie Roussos's par...   
4      Ukraine conflict: Oil price soars to highest l...   
...                                                  ...   
42110           Highlights: Wales make history in Dublin   
42111  Gang jailed over £200m of cocaine in banana boxes   
42112   Scottish Budget presents huge challenges for SNP   
42113  Celebrations as Wales make history qualifying ...   
42114  School tells Muslim girls it’s ‘not safe’ for ...   

                             pubDate  \
0      Mon, 07 Mar 2022 08:01:56 GMT   
1      Sun, 06 Mar 2022 22:49:58 GMT   
2      Mon, 07 Mar 2022 00:14:42 GMT   
3      Mon, 07 Mar 2022 00:05:40 GMT   
4      Mon, 07 Mar 2022 08:15:53 GMT   
...                              ...   

In [ ]:
# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)
tfidf = tfidf_vectorizer.fit_transform(df["cleaned_text"])
# NMF model
nmf_model = NMF(n_components=5, random_state=42)
nmf_topics = nmf_model.fit_transform(tfidf)
# Display top 10 words per topic
def display_topics(model, feature_names, top_words=10):
    topics = {}
    for idx, topic in enumerate(model.components_):
        top_features = [feature_names[i] for i in topic.argsort()[-top_words:][::-1]]
        topics[f"Topic {idx+1}"] = top_features
        print(f"Topic {idx+1}: {', '.join(top_features)}")
    return topics

nmf_topics_words = display_topics(nmf_model, tfidf_vectorizer.get_feature_names_out())


Topic 1: uk, people, year, bbc, new, minister, election, government, first, two
Topic 2: day, seven, past, closely, going, paying, attention, whats, test, youve
Topic 3: england, cup, world, league, win, final, manchester, first, womens, watch
Topic 4: say, police, attack, man, woman, official, ukraine, family, officer, president
Topic 5: selection, image, around, world, striking, taken, reader, past, seven, day


In [ ]:
# Count vectorizer for LDA
count_vectorizer = CountVectorizer(max_df=0.95, min_df=2)
count_data = count_vectorizer.fit_transform(df["cleaned_text"])
# LDA model
lda_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_topics = lda_model.fit_transform(count_data)
# Top 10 words per LDA topic
lda_topics_words = display_topics(lda_model, count_vectorizer.get_feature_names_out())


Topic 1: say, police, year, day, two, woman, death, said, bbc, former
Topic 2: say, bbc, government, uk, minister, new, party, leader, plan, labour
Topic 3: ukraine, say, russia, russian, president, price, city, year, war, ukrainian
Topic 4: world, england, cup, win, league, manchester, first, final, champion, watch
Topic 5: people, uk, year, first, election, one, show, new, found, say


In [ ]:
word1 = 'economy'
word2 = 'finance'
# Get first synset of each word
syn1 = wordnet.synsets(word1)[0]
syn2 = wordnet.synsets(word2)[0]
similarity = syn1.wup_similarity(syn2)
print(f"Wu-Palmer similarity between '{word1}' and '{word2}': {similarity}")


Wu-Palmer similarity between 'economy' and 'finance': 0.2857142857142857


In [ ]:
def jaccard_similarity(doc1, doc2):
    set1 = set(doc1.split())
    set2 = set(doc2.split())
    return len(set1 & set2) / len(set1 | set2)
# Select three documents
doc_indices = [0, 1, 2]
selected_docs = [df["cleaned_text"][i] for i in doc_indices]
# Compute pairwise similarities
pairs = list(combinations(range(3), 2))
for i, j in pairs:
    sim = jaccard_similarity(selected_docs[i], selected_docs[j])
    print(f"Jaccard similarity between Doc {i} & Doc {j}: {sim:.3f}")


Jaccard similarity between Doc 0 & Doc 1: 0.000
Jaccard similarity between Doc 0 & Doc 2: 0.053
Jaccard similarity between Doc 1 & Doc 2: 0.000
